# 12 Hashtag Association Recommendations

Automated hashtag association recommendation pipeline with reliability scoring and dashboard-ready outputs.

## 1. Imports And Setup
Load dependencies and project paths.

In [1]:
# !pip install mlxtend

from __future__ import annotations

import ast
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load Dataset
Search likely locations and choose the best available processed dataset.

In [2]:
def load_best_dataset(project_root: Path) -> tuple[pd.DataFrame, Path]:
    search_roots = [
        project_root / "data",
        project_root / "processed",
        project_root / "outputs",
        project_root / "reports",
        project_root,
    ]

    preferred_names = [
        "vanilla_processed.json",  # user-specified preferred dataset
        "kpi_dataset.json",
        "processed_posts.json",
        "processed_dataset.json",
        "final_dataset.json",
    ]

    candidates: list[Path] = []
    for root in search_roots:
        if root.exists():
            candidates.extend(root.rglob("*.json"))
            candidates.extend(root.rglob("*.csv"))

    # de-duplicate while preserving order
    seen = set()
    uniq = []
    for p in candidates:
        if p not in seen and ".venv" not in p.parts and "site-packages" not in p.parts:
            seen.add(p)
            uniq.append(p)

    # hard-prioritize explicit request if present
    explicit = project_root / "data" / "vanilla_processed.json"
    if explicit.exists():
        chosen = explicit
    else:
        scored: list[tuple[int, int, Path]] = []
        for p in uniq:
            name = p.name.lower()
            name_score = 1000
            for i, pref in enumerate(preferred_names):
                if name == pref:
                    name_score = i
                    break
            # heuristic recency and likely completeness
            recency = int(p.stat().st_mtime)
            scored.append((name_score, -recency, p))
        if not scored:
            raise FileNotFoundError("No candidate datasets found in project search paths.")
        scored.sort(key=lambda x: (x[0], x[1]))
        chosen = scored[0][2]

    if chosen.suffix.lower() == ".json":
        df = pd.read_json(chosen)
    elif chosen.suffix.lower() == ".csv":
        df = pd.read_csv(chosen)
    else:
        raise ValueError(f"Unsupported file type: {chosen}")

    return df, chosen


df_raw, selected_path = load_best_dataset(PROJECT_ROOT)
print(f"Dataset selected: {selected_path}")
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))

important_cols = [
    "business_name", "sector", "caption_text", "hashtags", "views_count", "followers_count",
    "likes_count", "comments_count", "post_type", "post_date", "view_rate", "engagement_rate"
]
for c in important_cols:
    if c in df_raw.columns:
        print(f"Missing {c}: {int(df_raw[c].isna().sum())}")
    else:
        print(f"Missing {c}: column_not_found")

print()
print("Preview:")
display(df_raw.head(3))


Dataset selected: C:\univ\Data mining\Project\data\vanilla_processed.json
Shape: (124, 22)
Columns: ['business_name', 'sector', 'followers_count', 'post_date', 'day_of_week', 'month', 'post_type', 'caption_text', 'caption_length', 'hashtags_count', 'emoji_count', 'likes_count', 'comments_count', 'views_count', 'language', 'CTA_present', 'promo_post', 'mentions_location', 'religious_theme', 'patriotic_theme', 'arabic_dialect_style', 'sponsored']
Missing business_name: 0
Missing sector: 0
Missing caption_text: 0
Missing hashtags: column_not_found
Missing views_count: 0
Missing followers_count: 0
Missing likes_count: 0
Missing comments_count: 0
Missing post_type: 0
Missing post_date: 0
Missing view_rate: column_not_found
Missing engagement_rate: column_not_found

Preview:


,business_name,sector,followers_count,post_date,day_of_week,month,post_type,caption_text,caption_length,hashtags_count,...,comments_count,views_count,language,CTA_present,promo_post,mentions_location,religious_theme,patriotic_theme,arabic_dialect_style,sponsored
0,Vanilla Palestine,Cafes/Restaurants,139000,2025-02-05,Wednesday,February,reel,شاركونا إجابتكم بالتعليقات 👇 #فانيلا_اليوم_وكل_يوم #vanilla_today_and_everyday,83,2,...,7,15200,Arabic,True,False,False,False,False,True,0
1,Vanilla Palestine,Cafes/Restaurants,139000,2025-02-04,Tuesday,February,reel,What’s your favorite cake ❤️🍰? #VANILLA_Today_and_everyday,65,1,...,0,11000,English,True,False,False,False,False,False,0
2,Vanilla Palestine,Cafes/Restaurants,139000,2025-02-01,Saturday,February,reel,كل يوم له بداية… وأجمل بداية بطاقة إيجابية #فانيلا_اليوم_وكل_يوم,72,1,...,1,12500,Arabic,False,False,False,False,False,True,0


## 3. Validate And Clean Core Columns
Ensure required columns exist and normalize key fields.

In [3]:
REQUIRED_COLS = [
    "business_name", "sector", "caption_text", "hashtags", "views_count", "followers_count",
    "likes_count", "comments_count", "post_type"
]


def normalize_post_type(x: Any) -> str:
    s = str(x).strip().lower()
    if s in {"image", "post", "img"}:
        return "post"
    if s in {"reel", "reels"}:
        return "reel"
    if s in {"video", "vid"}:
        return "video"
    if s in {"carousel", "album"}:
        return "carousel"
    return s if s else "unknown"


def ensure_core_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in REQUIRED_COLS:
        if c not in out.columns:
            out[c] = np.nan

    out["business_name"] = out["business_name"].fillna("Unknown Business").astype(str).str.strip()
    out["sector"] = out["sector"].fillna("Unknown").astype(str).str.strip()
    out["caption_text"] = out["caption_text"].fillna("").astype(str)
    out["post_type"] = out["post_type"].apply(normalize_post_type)

    numeric_cols = ["views_count", "followers_count", "likes_count", "comments_count"]
    for c in numeric_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    for c in ["views_count", "likes_count", "comments_count"]:
        out[c] = out[c].fillna(0).clip(lower=0)

    out["followers_count"] = out["followers_count"].fillna(0).clip(lower=0)

    return out


df_core = ensure_core_columns(df_raw)
print("Core-clean shape:", df_core.shape)
print(df_core[["views_count", "followers_count", "likes_count", "comments_count"]].describe().T)


Core-clean shape: (124, 23)
                 count           mean            std      min       25%       50%        75%        max
views_count      124.0  225736.290323  224737.148121   9300.0   56000.0  144500.0  327500.00  1000000.0
followers_count  124.0  128993.548387   20508.446572  87300.0  139000.0  139000.0  139000.00   139000.0
likes_count      124.0    1040.620968    1757.321477      4.0     303.0     769.0    1314.75    18600.0
comments_count   124.0     183.588710     486.473456      0.0       2.0      10.0      49.00     3300.0


## 4. Hashtag Extraction And Normalization
Parse hashtags from explicit column and/or caption text.

In [4]:
HASHTAG_RE = re.compile(r"(?<!\w)#([^\s#]+)", flags=re.UNICODE)


def normalize_hashtag(tag: Any) -> str:
    s = str(tag).strip()
    s = s.lstrip("#").strip()
    s = s.strip(".,!?;:'\"()[]{}<>")
    if not s:
        return ""
    # lowercase English while preserving Arabic text naturally
    s = s.lower()
    return s


def extract_hashtags(text: Any) -> list[str]:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return []
    matches = HASHTAG_RE.findall(str(text))
    out: list[str] = []
    seen = set()
    for m in matches:
        t = normalize_hashtag(m)
        if t and t not in seen:
            seen.add(t)
            out.append(t)
    return out


def parse_hashtags(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    if isinstance(value, list):
        raw = value
    elif isinstance(value, str):
        s = value.strip()
        if not s:
            raw = []
        else:
            # try safe literal parse for stringified list
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    raw = parsed
                else:
                    raw = extract_hashtags(s)
            except Exception:
                if "|" in s:
                    raw = [x.strip() for x in s.split("|") if x.strip()]
                elif "," in s:
                    raw = [x.strip() for x in s.split(",") if x.strip()]
                else:
                    raw = extract_hashtags(s)
    else:
        raw = [value]

    out: list[str] = []
    seen = set()
    for r in raw:
        t = normalize_hashtag(r)
        if t and t not in seen:
            seen.add(t)
            out.append(t)
    return out


def prepare_hashtags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    parsed_from_col = out["hashtags"].apply(parse_hashtags)
    parsed_from_caption = out["caption_text"].apply(extract_hashtags)

    merged = []
    for a, b in zip(parsed_from_col.tolist(), parsed_from_caption.tolist()):
        seen = set()
        combo = []
        for tag in a + b:
            if tag and tag not in seen:
                seen.add(tag)
                combo.append(tag)
        merged.append(combo)

    out["hashtags"] = merged
    out["hashtag_count"] = out["hashtags"].apply(len)
    return out


df_tags = prepare_hashtags(df_core)
posts_with_hashtags = int((df_tags["hashtag_count"] > 0).sum())
avg_hashtags = float(df_tags["hashtag_count"].mean())
print("Posts with >=1 hashtag:", posts_with_hashtags)
print("Average hashtags per post:", round(avg_hashtags, 3))

all_tags = [t for tags in df_tags["hashtags"] for t in tags]
tag_freq = pd.Series(all_tags, dtype="object").value_counts().rename_axis("hashtag").reset_index(name="frequency") if all_tags else pd.DataFrame(columns=["hashtag","frequency"])
print("Top 30 hashtags:")
display(tag_freq.head(30))

tag_freq.to_csv(REPORTS_DIR / "hashtag_frequency.csv", index=False)
print("Saved:", REPORTS_DIR / "hashtag_frequency.csv")


Posts with >=1 hashtag: 76
Average hashtags per post: 2.153
Top 30 hashtags:


,hashtag,frequency
0,فانيلا_اليوم_وكل_يوم,47
1,vanilla_today_and_everyday,34
2,vanilla,19
3,today_and_everyday,16
4,قهوة,8
5,كيك,8
6,فانيلا,8
7,coffee,8
8,cake,7
9,اليوم_وكل_يوم,7


Saved: C:\univ\Data mining\Project\reports\hashtag_frequency.csv


## 5. Create Performance Metrics And Labels
Build business-normalized and global-quantile labels.

In [5]:
def create_performance_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    denom = out["followers_count"].replace(0, np.nan)
    out["view_rate"] = out["views_count"] / denom
    out["engagement"] = out["likes_count"] + out["comments_count"]
    out["engagement_rate"] = out["engagement"] / denom

    out["view_rate"] = out["view_rate"].replace([np.inf, -np.inf], np.nan)
    out["engagement_rate"] = out["engagement_rate"].replace([np.inf, -np.inf], np.nan)

    out["business_median_view_rate"] = out.groupby("business_name")["view_rate"].transform("median")
    out["business_median_view_rate"] = out["business_median_view_rate"].replace(0, np.nan)
    out["view_rate_ratio"] = out["view_rate"] / out["business_median_view_rate"]

    def label_ratio(x: Any) -> str:
        if pd.isna(x):
            return "avg_view_rate"
        if x >= 1.5:
            return "high_view_rate"
        if x >= 0.8:
            return "avg_view_rate"
        return "low_view_rate"

    out["business_normalized_label"] = out["view_rate_ratio"].apply(label_ratio)

    # global quantile label (after trimming top 95% for binning only)
    valid_view = out["view_rate"].dropna()
    if len(valid_view) >= 3:
        q95 = valid_view.quantile(0.95)
        temp = out[out["view_rate"].le(q95) & out["view_rate"].notna()].copy()
        if len(temp) >= 3:
            try:
                temp["global_view_rate_label"] = pd.qcut(temp["view_rate"], q=3, labels=["low_view_rate", "avg_view_rate", "high_view_rate"])
            except ValueError:
                temp["global_view_rate_label"] = pd.qcut(temp["view_rate"].rank(method="first"), q=3, labels=["low_view_rate", "avg_view_rate", "high_view_rate"])
            out = out.merge(temp[["global_view_rate_label"]], left_index=True, right_index=True, how="left")
        else:
            out["global_view_rate_label"] = np.nan
    else:
        out["global_view_rate_label"] = np.nan

    out["performance_label"] = out["business_normalized_label"]
    return out


df_perf = create_performance_labels(df_tags)
print("Label counts:")
print(df_perf["performance_label"].value_counts(dropna=False))

sector_dist = pd.crosstab(df_perf["sector"], df_perf["performance_label"])
post_type_dist = pd.crosstab(df_perf["post_type"], df_perf["performance_label"])

print()
print("Label distribution by sector:")
display(sector_dist.head(20))
print()
print("Label distribution by post_type:")
display(post_type_dist)

label_dist = df_perf[["sector", "post_type", "performance_label"]].copy()
label_dist.to_csv(REPORTS_DIR / "performance_label_distribution.csv", index=False)
print("Saved:", REPORTS_DIR / "performance_label_distribution.csv")


Label counts:
performance_label
low_view_rate     50
high_view_rate    45
avg_view_rate     29
Name: count, dtype: int64

Label distribution by sector:


performance_label,avg_view_rate,high_view_rate,low_view_rate
sector,,,
Cafes/Restaurants,29,45,50



Label distribution by post_type:


performance_label,avg_view_rate,high_view_rate,low_view_rate
post_type,,,
post,0,0,14
reel,29,45,36


Saved: C:\univ\Data mining\Project\reports\performance_label_distribution.csv


## 6. Build Transactions
Create item transactions for association mining.

In [6]:
def build_transactions(df: pd.DataFrame, include_context: bool = True) -> tuple[list[list[str]], pd.DataFrame]:
    transactions: list[list[str]] = []
    rows = []
    for i, row in df.iterrows():
        tags = row.get("hashtags", [])
        if not isinstance(tags, list) or len(tags) == 0:
            continue
        label = row.get("performance_label")
        if pd.isna(label):
            continue

        items = [f"hashtag={t}" for t in tags] + [f"performance={label}"]
        if include_context:
            items += [
                f"post_type={row.get('post_type', 'unknown')}",
                f"sector={row.get('sector', 'Unknown')}",
            ]

        transactions.append(items)
        rows.append({
            "row_id": i,
            "transaction": items,
            "hashtag_count": len(tags),
            "performance_label": label,
        })

    tx_df = pd.DataFrame(rows)
    return transactions, tx_df


transactions, transactions_df = build_transactions(df_perf, include_context=True)
print("Valid transaction posts:", len(transactions))
print("Sample transactions:")
for t in transactions[:5]:
    print(t)


Valid transaction posts: 76
Sample transactions:
['hashtag=فانيلا_اليوم_وكل_يوم', 'hashtag=vanilla_today_and_everyday', 'performance=low_view_rate', 'post_type=reel', 'sector=Cafes/Restaurants']
['hashtag=vanilla_today_and_everyday', 'performance=low_view_rate', 'post_type=reel', 'sector=Cafes/Restaurants']
['hashtag=فانيلا_اليوم_وكل_يوم', 'performance=low_view_rate', 'post_type=reel', 'sector=Cafes/Restaurants']
['hashtag=فانيلا_اليوم_وكل_يوم', 'hashtag=vanilla_today_and_everyday', 'performance=avg_view_rate', 'post_type=reel', 'sector=Cafes/Restaurants']
['hashtag=فانيلا_اليوم_وكل_يوم', 'hashtag=طعام', 'hashtag=وزن', 'hashtag=أكل', 'hashtag=كيك', 'hashtag=today_and_everyday', 'hashtag=food', 'hashtag=eating', 'hashtag=cake', 'hashtag=desserts', 'performance=low_view_rate', 'post_type=reel', 'sector=Cafes/Restaurants']


## 7. Mine Frequent Itemsets And Rules
Run Apriori + association rules with adaptive thresholds and strict hashtag antecedent filtering.

In [7]:
def mine_hashtag_rules(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Any], int]:
    tx, _ = build_transactions(df, include_context=False)
    n_posts = len(tx)

    if n_posts < 100:
        min_support = 0.02
    elif n_posts < 500:
        min_support = 0.015
    else:
        min_support = 0.01

    min_confidence = 0.3

    diagnostics = {
        "n_posts": n_posts,
        "min_support": min_support,
        "min_confidence": min_confidence,
        "frequent_itemsets": 0,
        "raw_rules": 0,
        "filtered_rules": 0,
    }

    if n_posts == 0:
        return pd.DataFrame(), diagnostics, n_posts

    te = TransactionEncoder()
    ohe = pd.DataFrame(te.fit(tx).transform(tx), columns=te.columns_)

    itemsets = apriori(ohe, min_support=min_support, use_colnames=True)
    diagnostics["frequent_itemsets"] = int(len(itemsets))
    if itemsets.empty:
        return pd.DataFrame(), diagnostics, n_posts

    rules = association_rules(itemsets, metric="confidence", min_threshold=min_confidence)
    diagnostics["raw_rules"] = int(len(rules))
    if rules.empty:
        return pd.DataFrame(), diagnostics, n_posts

    rules = rules.copy()
    rules["antecedent_size"] = rules["antecedents"].apply(len)
    rules["consequent_size"] = rules["consequents"].apply(len)
    rules["consequent_item"] = rules["consequents"].apply(lambda s: next(iter(s)) if len(s) == 1 else None)

    valid_cons = {
        "performance=avg_view_rate",
        "performance=high_view_rate",
        "performance=low_view_rate",
    }

    mask = (
        (rules["consequent_size"] == 1)
        & (rules["consequent_item"].isin(valid_cons))
        & (rules["antecedent_size"] > 0)
        & (rules["antecedent_size"] <= 3)
        & (rules["antecedents"].apply(lambda s: all(str(x).startswith("hashtag=") for x in s)))
    )

    out = rules.loc[mask].copy()
    if out.empty:
        return out, diagnostics, n_posts

    out["count"] = (out["support"] * n_posts).round().astype(int)
    out["antecedent_count"] = (out["antecedent support"] * n_posts).round().astype(int)
    out["antecedents_str"] = out["antecedents"].apply(lambda s: "{" + ", ".join(sorted(s)) + "}")

    diagnostics["filtered_rules"] = int(len(out))
    return out.reset_index(drop=True), diagnostics, n_posts


## 8. Reliability Scoring
Assign reliability levels and recommendation scores.

In [8]:
def reliability_level(count: int, confidence: float) -> str:
    if count >= 10 and confidence >= 0.5:
        return "strong"
    if count >= 5 and confidence >= 0.6:
        return "medium"
    if count >= 3 and confidence >= 0.8:
        return "experimental"
    return "weak"


def score_rules(rules: pd.DataFrame) -> pd.DataFrame:
    if rules.empty:
        return rules.copy()

    out = rules.copy()
    out["reliability"] = [reliability_level(int(c), float(conf)) for c, conf in zip(out["count"], out["confidence"])]

    target_map = {
        "performance=high_view_rate": 1.0,
        "performance=avg_view_rate": 0.6,
        "performance=low_view_rate": -1.0,
    }
    out["target_score"] = out["consequent_item"].map(target_map).fillna(0.0)
    out["count_score"] = (out["count"] / 10).clip(upper=1)
    out["lift_score"] = (out["lift"] / 3).clip(upper=1)

    out["recommendation_score"] = (
        0.35 * out["confidence"]
        + 0.30 * out["lift_score"]
        + 0.25 * out["count_score"]
        + 0.10 * out["target_score"]
    )
    out["strength_score"] = out["confidence"] * out["lift"]
    return out


def explain_rule(row: pd.Series) -> str:
    tags = row.get("antecedents_str", "")
    consequent = str(row.get("consequent_item", "")).replace("performance=", "")
    count = int(row.get("count", 0))
    ante_count = int(row.get("antecedent_count", 0))
    conf = float(row.get("confidence", 0))
    lift = float(row.get("lift", 0))
    rel = row.get("reliability", "weak")

    base = (
        f"Try using {tags}. This pattern appeared in {ante_count} posts and {count} reached {consequent}. "
        f"Confidence: {conf*100:.1f}%, lift: {lift:.2f}x. Reliability: {rel}."
    )

    if consequent == "low_view_rate":
        return base + " This was associated with low_view_rate, but this is not proof of causality."
    if rel in {"weak", "experimental"}:
        return base + " Small sample size, treat as experiment."
    if rel == "medium":
        return base + " Treat this as promising but continue testing."
    return base + " Strong signal relative to the current dataset."


## 9. Generate Final Recommendation Tables
Create all rules, recommended, experimental, warning, and top recommendation outputs.

In [9]:
def _top_with_diversity(df: pd.DataFrame, k: int = 10) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    tmp = df.copy()
    tmp["antecedent_len"] = tmp["antecedents"].apply(len)
    tmp = tmp.sort_values(
        ["recommendation_score", "count", "confidence", "lift", "antecedent_len"],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)

    seen = set()
    kept = []
    for _, r in tmp.iterrows():
        key = tuple(sorted([x.replace("hashtag=", "") for x in r["antecedents"]]))
        if key in seen:
            continue
        seen.add(key)
        kept.append(r)
        if len(kept) >= k:
            break

    if not kept:
        return tmp.head(0)

    out = pd.DataFrame(kept)
    out["selection_reason"] = np.where(
        out["count"] >= out["count"].median(),
        "top_count",
        "top_strength",
    )
    return out


def generate_recommendation_text(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if out.empty:
        out["recommendation_text"] = []
        return out
    out["recommendation_text"] = out.apply(explain_rule, axis=1)
    return out


def generate_hashtag_recommendations(df: pd.DataFrame):
    all_rules, diagnostics, n_posts = mine_hashtag_rules(df)
    all_rules = score_rules(all_rules)

    if all_rules.empty:
        empty = pd.DataFrame()
        return {
            "diagnostics": diagnostics,
            "n_posts": n_posts,
            "all_rules": empty,
            "recommended_rules": empty,
            "experimental_rules": empty,
            "warning_rules": empty,
            "top_recommendations": empty,
        }, empty

    recommended_rules = all_rules[
        (all_rules["consequent_item"].isin(["performance=high_view_rate", "performance=avg_view_rate"]))
        & (all_rules["lift"] > 1.2)
        & ((all_rules["count"] >= 5) | ((all_rules["count"] >= 3) & (all_rules["confidence"] >= 0.8)))
    ].copy()

    experimental_rules = all_rules[
        (all_rules["count"].between(3, 4))
        & (all_rules["confidence"] >= 0.8)
        & (all_rules["lift"] > 1.2)
    ].copy()

    warning_rules = all_rules[
        (all_rules["consequent_item"] == "performance=low_view_rate")
        & (all_rules["lift"] > 1.2)
    ].copy()

    recommended_rules = recommended_rules.sort_values(
        ["recommendation_score", "count", "confidence", "lift"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    top_recommendations = _top_with_diversity(recommended_rules, k=10)

    all_rules = generate_recommendation_text(all_rules)
    recommended_rules = generate_recommendation_text(recommended_rules)
    experimental_rules = generate_recommendation_text(experimental_rules)
    warning_rules = generate_recommendation_text(warning_rules)
    top_recommendations = generate_recommendation_text(top_recommendations)

    outputs = {
        "diagnostics": diagnostics,
        "n_posts": n_posts,
        "all_rules": all_rules,
        "recommended_rules": recommended_rules,
        "experimental_rules": experimental_rules,
        "warning_rules": warning_rules,
        "top_recommendations": top_recommendations,
    }
    return outputs, all_rules


outputs, all_rules = generate_hashtag_recommendations(df_perf)

print("Rule diagnostics:", outputs["diagnostics"])
print("All mined rules:", len(outputs["all_rules"]))
print("Recommended rules:", len(outputs["recommended_rules"]))
print("Experimental rules:", len(outputs["experimental_rules"]))
print("Warning rules:", len(outputs["warning_rules"]))
print("Top recommendations:", len(outputs["top_recommendations"]))

print()
print("Top recommendation texts:")
for txt in outputs["top_recommendations"].get("recommendation_text", pd.Series([], dtype=str)).head(10).tolist():
    print("-", txt)


Rule diagnostics: {'n_posts': 76, 'min_support': 0.02, 'min_confidence': 0.3, 'frequent_itemsets': 842, 'raw_rules': 10936, 'filtered_rules': 277}
All mined rules: 277
Recommended rules: 16
Experimental rules: 13
Warning rules: 76
Top recommendations: 10

Top recommendation texts:
- Try using {hashtag=dessert, hashtag=vanilla}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, treat as experiment.
- Try using {hashtag=vanilla, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, treat as experiment.
- Try using {hashtag=dessert, hashtag=vanilla, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, treat as experiment.
- Try using {hashtag=dessert, hashtag=vanilla, hashtag=van

## 10. Save Report Outputs
Export all dashboard-ready CSV files into `reports/`.

In [10]:
save_map = {
    "hashtag_association_all_rules.csv": outputs["all_rules"],
    "hashtag_association_recommended_rules.csv": outputs["recommended_rules"],
    "hashtag_association_experimental_rules.csv": outputs["experimental_rules"],
    "hashtag_association_warning_rules.csv": outputs["warning_rules"],
    "hashtag_association_top_recommendations.csv": outputs["top_recommendations"],
}

for fname, frame in save_map.items():
    frame.to_csv(REPORTS_DIR / fname, index=False)
    print("Saved:", REPORTS_DIR / fname)


Saved: C:\univ\Data mining\Project\reports\hashtag_association_all_rules.csv


Saved: C:\univ\Data mining\Project\reports\hashtag_association_recommended_rules.csv
Saved: C:\univ\Data mining\Project\reports\hashtag_association_experimental_rules.csv
Saved: C:\univ\Data mining\Project\reports\hashtag_association_warning_rules.csv
Saved: C:\univ\Data mining\Project\reports\hashtag_association_top_recommendations.csv


## 11. Context-Specific Recommendations
Run subgroup mining for sector, post_type, and business_name with minimum row thresholds.

In [11]:
def generate_contextual_recommendations(df: pd.DataFrame) -> pd.DataFrame:
    contexts = [
        ("sector", 30),
        ("post_type", 30),
        ("business_name", 20),
    ]

    parts = []
    for context_col, min_rows in contexts:
        counts = df[context_col].value_counts(dropna=False)
        eligible = counts[counts >= min_rows].index.tolist()

        for v in eligible:
            sub = df[df[context_col] == v].copy()
            out, _ = generate_hashtag_recommendations(sub)
            rec = out["recommended_rules"].copy()
            if rec.empty:
                continue
            rec["context_type"] = context_col
            rec["context_value"] = str(v)
            parts.append(rec)

    if not parts:
        return pd.DataFrame()

    ctx = pd.concat(parts, ignore_index=True)
    ctx = ctx.sort_values(
        ["context_type", "context_value", "recommendation_score", "count", "confidence", "lift"],
        ascending=[True, True, False, False, False, False],
    ).reset_index(drop=True)
    return ctx


contextual_df = generate_contextual_recommendations(df_perf)
contextual_path = REPORTS_DIR / "hashtag_association_contextual_recommendations.csv"
contextual_df.to_csv(contextual_path, index=False)
print("Saved:", contextual_path)
print("Contextual rows:", len(contextual_df))

if not contextual_df.empty:
    print()
    print("Top recommendations per sector:")
    display(contextual_df[contextual_df["context_type"] == "sector"].groupby("context_value").head(3))

    print()
    print("Top recommendations per post_type:")
    display(contextual_df[contextual_df["context_type"] == "post_type"].groupby("context_value").head(3))

    print()
    print("Top recommendations per business_name:")
    display(contextual_df[contextual_df["context_type"] == "business_name"].groupby("context_value").head(3))


Saved: C:\univ\Data mining\Project\reports\hashtag_association_contextual_recommendations.csv
Contextual rows: 54

Top recommendations per sector:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,...,antecedents_str,reliability,target_score,count_score,lift_score,recommendation_score,strength_score,recommendation_text,context_type,context_value
38,"(hashtag=dessert, hashtag=vanilla)",(performance=high_view_rate),0.052632,0.302632,0.052632,1.0,3.304348,0.036704,inf,0.736111,...,"{hashtag=dessert, hashtag=vanilla}",experimental,1.0,0.4,1.0,0.85,3.304348,"Try using {hashtag=dessert, hashtag=vanilla}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, treat as...",sector,Cafes/Restaurants
39,"(hashtag=vanilla_cake, hashtag=vanilla)",(performance=high_view_rate),0.052632,0.302632,0.052632,1.0,3.304348,0.036704,inf,0.736111,...,"{hashtag=vanilla, hashtag=vanilla_cake}",experimental,1.0,0.4,1.0,0.85,3.304348,"Try using {hashtag=vanilla, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, tre...",sector,Cafes/Restaurants
40,"(hashtag=dessert, hashtag=vanilla_cake, hashtag=vanilla)",(performance=high_view_rate),0.052632,0.302632,0.052632,1.0,3.304348,0.036704,inf,0.736111,...,"{hashtag=dessert, hashtag=vanilla, hashtag=vanilla_cake}",experimental,1.0,0.4,1.0,0.85,3.304348,"Try using {hashtag=dessert, hashtag=vanilla, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small...",sector,Cafes/Restaurants



Top recommendations per post_type:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,...,antecedents_str,reliability,target_score,count_score,lift_score,recommendation_score,strength_score,recommendation_text,context_type,context_value
16,(hashtag=dessert),(performance=high_view_rate),0.060606,0.348485,0.060606,1.0,2.869565,0.039486,inf,0.693548,...,{hashtag=dessert},experimental,1.0,0.4,0.956522,0.836957,2.869565,"Try using {hashtag=dessert}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 2.87x. Reliability: experimental. Small sample size, treat as experiment.",post_type,reel
17,"(hashtag=dessert, hashtag=vanilla)",(performance=high_view_rate),0.060606,0.348485,0.060606,1.0,2.869565,0.039486,inf,0.693548,...,"{hashtag=dessert, hashtag=vanilla}",experimental,1.0,0.4,0.956522,0.836957,2.869565,"Try using {hashtag=dessert, hashtag=vanilla}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 2.87x. Reliability: experimental. Small sample size, treat as...",post_type,reel
18,"(hashtag=dessert, hashtag=vanilla_cake)",(performance=high_view_rate),0.060606,0.348485,0.060606,1.0,2.869565,0.039486,inf,0.693548,...,"{hashtag=dessert, hashtag=vanilla_cake}",experimental,1.0,0.4,0.956522,0.836957,2.869565,"Try using {hashtag=dessert, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 2.87x. Reliability: experimental. Small sample size, tre...",post_type,reel



Top recommendations per business_name:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,...,antecedents_str,reliability,target_score,count_score,lift_score,recommendation_score,strength_score,recommendation_text,context_type,context_value
0,"(hashtag=dessert, hashtag=vanilla)",(performance=high_view_rate),0.052632,0.302632,0.052632,1.0,3.304348,0.036704,inf,0.736111,...,"{hashtag=dessert, hashtag=vanilla}",experimental,1.0,0.4,1.0,0.85,3.304348,"Try using {hashtag=dessert, hashtag=vanilla}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, treat as...",business_name,Vanilla Palestine
1,"(hashtag=vanilla_cake, hashtag=vanilla)",(performance=high_view_rate),0.052632,0.302632,0.052632,1.0,3.304348,0.036704,inf,0.736111,...,"{hashtag=vanilla, hashtag=vanilla_cake}",experimental,1.0,0.4,1.0,0.85,3.304348,"Try using {hashtag=vanilla, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, tre...",business_name,Vanilla Palestine
2,"(hashtag=dessert, hashtag=vanilla_cake, hashtag=vanilla)",(performance=high_view_rate),0.052632,0.302632,0.052632,1.0,3.304348,0.036704,inf,0.736111,...,"{hashtag=dessert, hashtag=vanilla, hashtag=vanilla_cake}",experimental,1.0,0.4,1.0,0.85,3.304348,"Try using {hashtag=dessert, hashtag=vanilla, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small...",business_name,Vanilla Palestine


## 12. Reusable Function Entry Point
Simple usage and preview of recommendation table.

In [12]:
recommendations = outputs["top_recommendations"].copy()

preview_cols = [
    "antecedents_str",
    "consequent_item",
    "count",
    "antecedent_count",
    "support",
    "confidence",
    "lift",
    "reliability",
    "recommendation_score",
    "recommendation_text",
]

if recommendations.empty:
    print("No top recommendations produced under current constraints.")
else:
    display(recommendations[preview_cols].head(20))


,antecedents_str,consequent_item,count,antecedent_count,support,confidence,lift,reliability,recommendation_score,recommendation_text
0,"{hashtag=dessert, hashtag=vanilla}",performance=high_view_rate,4,4,0.052632,1.000,3.304348,experimental,0.850000,"Try using {hashtag=dessert, hashtag=vanilla}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, treat as..."
1,"{hashtag=vanilla, hashtag=vanilla_cake}",performance=high_view_rate,4,4,0.052632,1.000,3.304348,experimental,0.850000,"Try using {hashtag=vanilla, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small sample size, tre..."
2,"{hashtag=dessert, hashtag=vanilla, hashtag=vanilla_cake}",performance=high_view_rate,4,4,0.052632,1.000,3.304348,experimental,0.850000,"Try using {hashtag=dessert, hashtag=vanilla, hashtag=vanilla_cake}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: experimental. Small..."
3,"{hashtag=dessert, hashtag=vanilla, hashtag=vanilla_today_and_everyday}",performance=high_view_rate,4,4,0.052632,1.000,3.304348,experimental,0.850000,"Try using {hashtag=dessert, hashtag=vanilla, hashtag=vanilla_today_and_everyday}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: exper..."
4,"{hashtag=vanilla, hashtag=vanilla_cake, hashtag=vanilla_today_and_everyday}",performance=high_view_rate,4,4,0.052632,1.000,3.304348,experimental,0.850000,"Try using {hashtag=vanilla, hashtag=vanilla_cake, hashtag=vanilla_today_and_everyday}. This pattern appeared in 4 posts and 4 reached high_view_rate. Confidence: 100.0%, lift: 3.30x. Reliability: ..."
5,"{hashtag=vanilla, hashtag=vanilla_today_and_everyday}",performance=high_view_rate,4,5,0.052632,0.800,2.643478,experimental,0.744348,"Try using {hashtag=vanilla, hashtag=vanilla_today_and_everyday}. This pattern appeared in 5 posts and 4 reached high_view_rate. Confidence: 80.0%, lift: 2.64x. Reliability: experimental. Small sam..."
6,{hashtag=coffee},performance=avg_view_rate,5,8,0.065789,0.625,3.392857,medium,0.703750,"Try using {hashtag=coffee}. This pattern appeared in 8 posts and 5 reached avg_view_rate. Confidence: 62.5%, lift: 3.39x. Reliability: medium. Treat this as promising but continue testing."
7,{hashtag=قهوة},performance=avg_view_rate,5,8,0.065789,0.625,3.392857,medium,0.703750,"Try using {hashtag=قهوة}. This pattern appeared in 8 posts and 5 reached avg_view_rate. Confidence: 62.5%, lift: 3.39x. Reliability: medium. Treat this as promising but continue testing."
8,"{hashtag=coffee, hashtag=today_and_everyday}",performance=avg_view_rate,5,8,0.065789,0.625,3.392857,medium,0.703750,"Try using {hashtag=coffee, hashtag=today_and_everyday}. This pattern appeared in 8 posts and 5 reached avg_view_rate. Confidence: 62.5%, lift: 3.39x. Reliability: medium. Treat this as promising b..."
9,"{hashtag=coffee, hashtag=قهوة}",performance=avg_view_rate,5,8,0.065789,0.625,3.392857,medium,0.703750,"Try using {hashtag=coffee, hashtag=قهوة}. This pattern appeared in 8 posts and 5 reached avg_view_rate. Confidence: 62.5%, lift: 3.39x. Reliability: medium. Treat this as promising but continue te..."


## 13. Visual Inspection Outputs
Quick plots/tables for hashtag frequency, label counts, recommendation ranking, and confidence/count diagnostics.

In [13]:
# Top 20 hashtag frequency table + plot
freq_top20 = tag_freq.head(20)
display(freq_top20)

if not freq_top20.empty:
    plt.figure(figsize=(10, 6))
    plt.barh(freq_top20["hashtag"].iloc[::-1], freq_top20["frequency"].iloc[::-1])
    plt.title("Top 20 Hashtags Frequency")
    plt.xlabel("Frequency")
    plt.tight_layout()
    plt.show()

# Performance label counts
label_counts = df_perf["performance_label"].value_counts().rename_axis("performance_label").reset_index(name="count")
display(label_counts)

# Top 15 recommendations by score
top15 = outputs["recommended_rules"].sort_values("recommendation_score", ascending=False).head(15)
display(top15[["antecedents_str", "consequent_item", "recommendation_score", "count", "confidence", "lift", "reliability"]])

if not top15.empty:
    plt.figure(figsize=(10, 5))
    plt.barh(top15["antecedents_str"].iloc[::-1], top15["recommendation_score"].iloc[::-1])
    plt.title("Top 15 Recommendations By Score")
    plt.xlabel("Recommendation Score")
    plt.tight_layout()
    plt.show()

# Confidence vs count scatter
if not outputs["all_rules"].empty:
    plt.figure(figsize=(7, 5))
    plt.scatter(outputs["all_rules"]["count"], outputs["all_rules"]["confidence"], alpha=0.6)
    plt.title("Confidence vs Count")
    plt.xlabel("Count")
    plt.ylabel("Confidence")
    plt.tight_layout()
    plt.show()


,hashtag,frequency
0,فانيلا_اليوم_وكل_يوم,47
1,vanilla_today_and_everyday,34
2,vanilla,19
3,today_and_everyday,16
4,قهوة,8
5,كيك,8
6,فانيلا,8
7,coffee,8
8,cake,7
9,اليوم_وكل_يوم,7


C:\Users\LEGION\AppData\Local\Temp\ipykernel_21556\625546782.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,performance_label,count
0,low_view_rate,50
1,high_view_rate,45
2,avg_view_rate,29


,antecedents_str,consequent_item,recommendation_score,count,confidence,lift,reliability
0,"{hashtag=dessert, hashtag=vanilla}",performance=high_view_rate,0.850000,4,1.000000,3.304348,experimental
1,"{hashtag=vanilla, hashtag=vanilla_cake}",performance=high_view_rate,0.850000,4,1.000000,3.304348,experimental
2,"{hashtag=dessert, hashtag=vanilla, hashtag=vanilla_cake}",performance=high_view_rate,0.850000,4,1.000000,3.304348,experimental
3,"{hashtag=dessert, hashtag=vanilla, hashtag=vanilla_today_and_everyday}",performance=high_view_rate,0.850000,4,1.000000,3.304348,experimental
4,"{hashtag=vanilla, hashtag=vanilla_cake, hashtag=vanilla_today_and_everyday}",performance=high_view_rate,0.850000,4,1.000000,3.304348,experimental
5,"{hashtag=vanilla, hashtag=vanilla_today_and_everyday}",performance=high_view_rate,0.744348,4,0.800000,2.643478,experimental
6,{hashtag=coffee},performance=avg_view_rate,0.703750,5,0.625000,3.392857,medium
7,{hashtag=قهوة},performance=avg_view_rate,0.703750,5,0.625000,3.392857,medium
8,"{hashtag=coffee, hashtag=today_and_everyday}",performance=avg_view_rate,0.703750,5,0.625000,3.392857,medium
9,"{hashtag=coffee, hashtag=قهوة}",performance=avg_view_rate,0.703750,5,0.625000,3.392857,medium


C:\Users\LEGION\AppData\Local\Temp\ipykernel_21556\625546782.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\LEGION\AppData\Local\Temp\ipykernel_21556\625546782.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Quality Checks
Diagnostics to verify robustness and filtering behavior.

In [14]:
print("Enough hashtag posts:", posts_with_hashtags, "(threshold check >= 30)")
print("Total rules generated:", outputs["diagnostics"].get("raw_rules", 0))
print("Rules after filtering:", outputs["diagnostics"].get("filtered_rules", 0))
print("Reliability counts:")
if outputs["all_rules"].empty:
    print("No rules available")
else:
    print(outputs["all_rules"]["reliability"].value_counts())

if outputs["recommended_rules"].empty:
    print("No final recommendations available")
else:
    min_count_final = outputs["recommended_rules"]["count"].min()
    print("Min count in final recommendations:", min_count_final)
    print("Any final recommendation with count < 3:", bool((outputs["recommended_rules"]["count"] < 3).any()))

    tiny_high_lift = outputs["all_rules"][(outputs["all_rules"]["count"] < 3) & (outputs["all_rules"]["lift"] > 2.0)]
    print("High-lift tiny-count rule candidates:", len(tiny_high_lift))
    if not tiny_high_lift.empty:
        display(tiny_high_lift[["antecedents_str", "consequent_item", "count", "confidence", "lift"]].head(20))


Enough hashtag posts: 76 (threshold check >= 30)
Total rules generated: 10936
Rules after filtering: 277
Reliability counts:
reliability
weak            255
experimental     13
medium            8
strong            1
Name: count, dtype: int64
Min count in final recommendations: 4
Any final recommendation with count < 3: False
High-lift tiny-count rule candidates: 71


,antecedents_str,consequent_item,count,confidence,lift
8,{hashtag=happymothersday},performance=high_view_rate,2,0.666667,2.202899
9,{hashtag=morning},performance=avg_view_rate,2,1.000000,5.428571
10,{hashtag=mothersday},performance=high_view_rate,2,0.666667,2.202899
17,{hashtag=todayandeveryday},performance=high_view_rate,2,1.000000,3.304348
24,{hashtag=الصباح},performance=avg_view_rate,2,1.000000,5.428571
35,{hashtag=يوم_الأم},performance=high_view_rate,2,0.666667,2.202899
43,"{hashtag=cake, hashtag=فانيلا_اليوم_وكل_يوم}",performance=avg_view_rate,2,0.400000,2.171429
49,"{hashtag=coffee, hashtag=morning}",performance=avg_view_rate,2,1.000000,5.428571
52,"{hashtag=coffee, hashtag=vanilla}",performance=avg_view_rate,2,0.666667,3.619048
53,"{hashtag=coffee, hashtag=الصباح}",performance=avg_view_rate,2,1.000000,5.428571


## 15. Final Summary
High-level run summary.

In [15]:
strong_n = int((outputs["recommended_rules"].get("reliability", pd.Series([], dtype=str)) == "strong").sum()) if not outputs["recommended_rules"].empty else 0
medium_n = int((outputs["recommended_rules"].get("reliability", pd.Series([], dtype=str)) == "medium").sum()) if not outputs["recommended_rules"].empty else 0
experimental_n = int((outputs["recommended_rules"].get("reliability", pd.Series([], dtype=str)) == "experimental").sum()) if not outputs["recommended_rules"].empty else 0

print(f"Dataset used: {selected_path}")
print(f"Valid posts used: {outputs['n_posts']}")
print(f"Posts with hashtags: {posts_with_hashtags}")
print(f"Total rules mined: {len(outputs['all_rules'])}")
print(f"Final recommendations: {len(outputs['recommended_rules'])}")
print(f"Strong recommendations: {strong_n}")
print(f"Medium recommendations: {medium_n}")
print(f"Experimental recommendations: {experimental_n}")
print("Saved outputs to reports/")


Dataset used: C:\univ\Data mining\Project\data\vanilla_processed.json
Valid posts used: 76
Posts with hashtags: 76
Total rules mined: 277
Final recommendations: 16
Strong recommendations: 0
Medium recommendations: 6
Experimental recommendations: 6
Saved outputs to reports/
